# Ingesta de Datos de Fraude en Hadoop (HDFS) con PySpark

**Objetivo:** Descargar el dataset de transacciones financieras nigerianas desde HuggingFace,  
cargarlo con PySpark, aplicar transformaciones básicas y persistirlo en HDFS.

**Dataset:** `electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset`  
**HDFS paths:**  
- Raw  → `hdfs://hadoop:9000/data/raw/transactions.parquet`  
- Processed → `hdfs://hadoop:9000/data/processed/transactions_clean.parquet`  
- Results → `hdfs://hadoop:9000/data/fraud-results/`

> Este notebook se ejecuta desde **JupyterLab** (`ml-env`, puerto 8888).  
> El contenedor Hadoop debe estar levantado: `docker compose -f infrastructure/hadoop/docker-compose.yml up -d`

## 1. Instalación de Dependencias

In [1]:
# Solo necesario la primera vez en el entorno ml-env
import subprocess, sys

pkgs = ["pyspark==3.5.3", "pyarrow==16.1.0", "datasets==2.20.0", "huggingface_hub==0.23.4"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", pkg], check=True)

print("Dependencias instaladas correctamente.")

Dependencias instaladas correctamente.


## 2. Descargar dataset desde HuggingFace

In [2]:
from datasets import load_dataset
import pandas as pd

# Tamaño de muestra para el pipeline de desarrollo.
# El modelo fue entrenado con el dataset completo; aquí solo demostramos
# el flujo Spark → HDFS. Con 100 k filas el pipeline completa en ~1 min.
SAMPLE_SIZE = 100_000

print("Descargando dataset de HuggingFace...")
dataset = load_dataset(
    "electricsheepafrica/Nigerian-Financial-Transactions-and-Fraud-Detection-Dataset",
    trust_remote_code=True
)

df_full = dataset["train"].to_pandas()
print(f"Dataset completo: {len(df_full):,} filas | {df_full.columns.tolist()}")

# Muestreo estratificado por clase para mantener proporción de fraude
if len(df_full) > SAMPLE_SIZE:
    fraud_col = next((c for c in df_full.columns if "fraud" in c.lower() or "label" in c.lower()), None)
    if fraud_col:
        df_raw = (
            df_full.groupby(fraud_col, group_keys=False)
                   .apply(lambda g: g.sample(frac=SAMPLE_SIZE / len(df_full), random_state=42))
                   .reset_index(drop=True)
        )
    else:
        df_raw = df_full.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
    print(f"[INFO] Muestra de desarrollo: {len(df_raw):,} filas ({100*len(df_raw)/len(df_full):.1f}% del total)")
else:
    df_raw = df_full

df_raw.head(3)


/usr/local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Descargando dataset de HuggingFace...
Dataset completo: 5,000,000 filas | ['transaction_id', 'timestamp', 'sender_account', 'receiver_account', 'transaction_type', 'merchant_category', 'location', 'device_used', 'is_fraud', 'fraud_type', 'time_since_last_transaction', 'spending_deviation_score', 'velocity_score', 'geo_anomaly_score', 'payment_channel', 'ip_address', 'device_hash', 'amount_ngn', 'bvn_linked', 'new_device_transaction', 'sender_persona', 'geospatial_velocity_anomaly', 'txn_hour', 'is_weekend', 'is_salary_week', 'is_night_txn', 'device_seen_count', 'is_device_shared', 'ip_seen_count', 'is_ip_shared', 'user_txn_count_total', 'user_avg_txn_amt', 'user_std_txn_amt', 'user_txn_frequency_24h', 'user_top_category', 'txn_count_last_1h', 'txn_count_last_24h', 'total_amount_last_1h', 'time_since_last', 'avg_gap_between_txns', 'merchant_fraud_rate', 'channel_risk_score', 'persona_fraud_risk', 'location_fraud_risk', 'ip_geo_region']


/tmp/ipykernel_366/2869435496.py:24: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(frac=SAMPLE_SIZE / len(df_full), random_state=42))


[INFO] Muestra de desarrollo: 100,000 filas (2.0% del total)


,transaction_id,timestamp,sender_account,receiver_account,transaction_type,merchant_category,location,device_used,is_fraud,fraud_type,...,txn_count_last_1h,txn_count_last_24h,total_amount_last_1h,time_since_last,avg_gap_between_txns,merchant_fraud_rate,channel_risk_score,persona_fraud_risk,location_fraud_risk,ip_geo_region
0,T2206966,2023-12-02 05:59:42.340373,3485408998,1249563455,deposit,Konga Order,Lagos,pos,False,None,...,4,4,557776.25,114227.023683,99530.660941,0.034949,0.3,0.5,0.036446,South West
1,T668947,2023-05-28 10:55:09.729421,9863353709,3797866830,payment,Bet9ja Stake,Onitsha,atm,False,None,...,5,5,1028871.27,6285.627762,25599.657338,0.035642,0.6,0.5,0.036054,South East
2,T1761976,2023-09-28 16:55:37.786585,1294827147,8103769122,transfer,Other Transaction,Kano,atm,False,None,...,9,9,808058.46,33242.698175,39383.919093,0.036505,0.6,0.5,0.035558,North West


## 3. Inicializar SparkSession conectada a HDFS

In [3]:
import os
import socket
from pyspark.sql import SparkSession

# --- Detección automática de HDFS ---
HDFS_URI = "hdfs://hadoop:9000"
LOCAL_DATA = "/app/data"   # volumen montado en ml-env → ../../data

def _hdfs_available(host="hadoop", port=9000, timeout=4):
    try:
        socket.create_connection((host, port), timeout=timeout).close()
        return True
    except OSError:
        return False

HDFS_UP = _hdfs_available()
if HDFS_UP:
    DATA_ROOT = HDFS_URI
    print(f"[OK] HDFS disponible → almacenamiento en {DATA_ROOT}")
else:
    DATA_ROOT = f"file://{LOCAL_DATA}"
    print(f"[WARN] Hadoop no detectado → modo local: {DATA_ROOT}")
    print("       Para usar HDFS: docker compose -f infrastructure/hadoop/docker-compose.yml up -d")

# --- SparkSession ---
builder = (
    SparkSession.builder
    .appName("FraudDetection-Ingesta")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    # 4 g para el driver: necesario para shuffle/features sobre 5 M filas en local[*]
    .config("spark.driver.memory", "4g")
    .config("spark.driver.maxResultSize", "2g")
    # Arrow: pandas→Spark vectorizado (se usa en celdas de features)
    .config("spark.sql.execution.arrow.pyspark.enabled", "true")
    # Evitar OOM en operaciones de sort/join grandes
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "50000")
)

if HDFS_UP:
    builder = (
        builder
        .config("spark.hadoop.fs.defaultFS", HDFS_URI)
        .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
        .config("spark.hadoop.ipc.client.connect.timeout", "10000")
        .config("spark.hadoop.ipc.client.connect.max.retries.on.timeouts", "3")
        .config("spark.hadoop.dfs.client.socket-timeout", "30000")
    )

spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")
print(f"Spark {spark.version} iniciado  |  DATA_ROOT = {DATA_ROOT}")


[OK] HDFS disponible → almacenamiento en hdfs://hadoop:9000


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/20 16:50:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3 iniciado  |  DATA_ROOT = hdfs://hadoop:9000


## 4. Cargar pandas → Spark DataFrame y guardar en HDFS (raw)

In [4]:
import os
from pyspark.sql.types import *

TMP_PARQUET = "/tmp/df_raw_ingesta.parquet"

print(f"Escribiendo {len(df_raw):,} filas × {len(df_raw.columns)} columnas a parquet temporal...")
df_raw.to_parquet(TMP_PARQUET, index=False, engine="pyarrow")
print(f"[OK] Parquet temporal: {TMP_PARQUET}  ({os.path.getsize(TMP_PARQUET)/1024/1024:.1f} MB)")

# IMPORTANTE: prefijo file:// explícito para que Spark no lo reinterprete como
# hdfs://hadoop:9000/tmp/... (que ocurre cuando defaultFS apunta a HDFS)
sdf_raw = spark.read.parquet(f"file://{TMP_PARQUET}")
print(f"Spark DataFrame cargado: {len(df_raw):,} filas, {len(sdf_raw.columns)} columnas")
sdf_raw.printSchema()

# Guardar raw en HDFS (o local si Hadoop no está disponible)
RAW_PATH = f"{DATA_ROOT}/raw/transactions.parquet"
if not HDFS_UP:
    os.makedirs(f"{LOCAL_DATA}/raw", exist_ok=True)

sdf_raw.write.mode("overwrite").parquet(RAW_PATH)
print(f"[OK] Dataset raw guardado en {RAW_PATH}")

# Limpiar el temporal
os.remove(TMP_PARQUET)
print(f"[OK] Temporal eliminado")


Escribiendo 100,000 filas × 45 columnas a parquet temporal...
[OK] Parquet temporal: /tmp/df_raw_ingesta.parquet  (14.5 MB)


Spark DataFrame cargado: 100,000 filas, 45 columnas
root
 |-- transaction_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- sender_account: long (nullable = true)
 |-- receiver_account: long (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- location: string (nullable = true)
 |-- device_used: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- fraud_type: string (nullable = true)
 |-- time_since_last_transaction: double (nullable = true)
 |-- spending_deviation_score: double (nullable = true)
 |-- velocity_score: long (nullable = true)
 |-- geo_anomaly_score: double (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- device_hash: string (nullable = true)
 |-- amount_ngn: double (nullable = true)
 |-- bvn_linked: boolean (nullable = true)
 |-- new_device_transaction: boolean (nullable = true)
 |-- sender_persona: st

26/05/20 16:50:13 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


[OK] Dataset raw guardado en hdfs://hadoop:9000/raw/transactions.parquet
[OK] Temporal eliminado


## 5. Transformaciones y limpieza de datos

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col, when, isnan, count
from pyspark.sql.types import DoubleType, FloatType

# Leer desde HDFS (o local)
sdf = spark.read.parquet(RAW_PATH)

# isnan() solo funciona en float/double — para booleanos, strings e integers
# basta con isNull(). Detectamos el tipo antes de aplicarlo.
def null_expr(c_name):
    dtype = sdf.schema[c_name].dataType
    if isinstance(dtype, (DoubleType, FloatType)):
        return count(when(col(c_name).isNull() | isnan(c_name), c_name)).alias(c_name)
    return count(when(col(c_name).isNull(), c_name)).alias(c_name)

print("=== Nulos por columna ===")
sdf.select([null_expr(c) for c in sdf.columns]).show(vertical=True)

# Columnas críticas adaptadas al dataset nigeriano
critical_cols = ["amount_ngn", "transaction_type", "is_fraud"]
sdf_clean = sdf.dropna(subset=[c for c in critical_cols if c in sdf.columns])

# Normalizar tipo de transacción (acepta "transaction_type" o "type")
type_col = next((c for c in ["transaction_type", "type"] if c in sdf_clean.columns), None)
if type_col:
    sdf_clean = sdf_clean.withColumn(type_col, F.upper(F.trim(col(type_col))))

# Filtrar importes negativos o cero
amount_col = next((c for c in ["amount_ngn", "amount"] if c in sdf_clean.columns), None)
if amount_col:
    sdf_clean = sdf_clean.filter(col(amount_col) > 0)

print(f"Filas tras limpieza: {sdf_clean.count():,}")
# vertical=True: muestra una fila por bloque → legible con 45 columnas
sdf_clean.show(3, vertical=True, truncate=50)


=== Nulos por columna ===


26/05/20 16:50:20 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


-RECORD 0----------------------------
 transaction_id              | 0     
 timestamp                   | 0     
 sender_account              | 0     
 receiver_account            | 0     
 transaction_type            | 0     
 merchant_category           | 0     
 location                    | 0     
 device_used                 | 0     
 is_fraud                    | 0     
 fraud_type                  | 96409 
 time_since_last_transaction | 18058 
 spending_deviation_score    | 0     
 velocity_score              | 0     
 geo_anomaly_score           | 0     
 payment_channel             | 0     
 ip_address                  | 0     
 device_hash                 | 0     
 amount_ngn                  | 0     
 bvn_linked                  | 0     
 new_device_transaction      | 0     
 sender_persona              | 0     
 geospatial_velocity_anomaly | 0     
 txn_hour                    | 0     
 is_weekend                  | 0     
 is_salary_week              | 0     
 is_night_tx

## 6. Feature engineering para el modelo de fraude

In [6]:
from pyspark.sql.functions import lit

# One-hot encoding manual del tipo de transacción
tipos = ["CASH_IN", "CASH_OUT", "DEBIT", "PAYMENT", "TRANSFER"]

sdf_features = sdf_clean
for t in tipos:
    col_name = f"type_{t}"
    if "type" in sdf_features.columns:
        sdf_features = sdf_features.withColumn(
            col_name,
            when(col("type") == t, 1.0).otherwise(0.0)
        )
    else:
        sdf_features = sdf_features.withColumn(col_name, lit(0.0))

# Renombrar columnas al formato esperado por el modelo (igual que el API)
rename_map = {
    "oldbalanceOrg":  "old_balance_orig",
    "newbalanceOrig": "new_balance_orig",
    "oldbalanceDest": "old_balance_dest",
    "newbalanceDest": "new_balance_dest",
    "isFraud":        "label"
}
for old_name, new_name in rename_map.items():
    if old_name in sdf_features.columns:
        sdf_features = sdf_features.withColumnRenamed(old_name, new_name)

# Columnas de grafos (PageRank, comunidades) — placeholders para Neo4j
for graph_col in ["orig_out_degree", "orig_pagerank", "orig_community",
                  "dest_in_degree", "dest_pagerank", "dest_community"]:
    if graph_col not in sdf_features.columns:
        sdf_features = sdf_features.withColumn(graph_col, lit(0.0))

print(f"Features preparadas: {len(sdf_features.columns)} columnas")
sdf_features.printSchema()

Features preparadas: 56 columnas
root
 |-- transaction_id: string (nullable = true)
 |-- timestamp: string (nullable = true)
 |-- sender_account: long (nullable = true)
 |-- receiver_account: long (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- merchant_category: string (nullable = true)
 |-- location: string (nullable = true)
 |-- device_used: string (nullable = true)
 |-- is_fraud: boolean (nullable = true)
 |-- fraud_type: string (nullable = true)
 |-- time_since_last_transaction: double (nullable = true)
 |-- spending_deviation_score: double (nullable = true)
 |-- velocity_score: long (nullable = true)
 |-- geo_anomaly_score: double (nullable = true)
 |-- payment_channel: string (nullable = true)
 |-- ip_address: string (nullable = true)
 |-- device_hash: string (nullable = true)
 |-- amount_ngn: double (nullable = true)
 |-- bvn_linked: boolean (nullable = true)
 |-- new_device_transaction: boolean (nullable = true)
 |-- sender_persona: string (nullable = tr

## 7. Guardar datos procesados en HDFS

In [ ]:
import os

PROCESSED_PATH = f"{DATA_ROOT}/data/processed/transactions_clean.parquet"

if not HDFS_UP:
    os.makedirs(f"{LOCAL_DATA}/data/processed", exist_ok=True)

sdf_features.write.mode("overwrite").parquet(PROCESSED_PATH)
print(f"[OK] Datos procesados guardados en {PROCESSED_PATH}")

# Verificar uso de espacio usando la API Java de Spark (no requiere cliente hdfs instalado)
if HDFS_UP:
    try:
        fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
        summary = fs.getContentSummary(spark._jvm.org.apache.hadoop.fs.Path("/data/"))
        size_mb = summary.getLength() / 1024 / 1024
        print(f"Uso de espacio HDFS /data/: {size_mb:.1f} MB")
    except Exception as e:
        print(f"[WARN] No se pudo consultar espacio HDFS: {e}")


[OK] Datos procesados guardados en hdfs://hadoop:9000/data/processed/transactions_clean.parquet


FileNotFoundError: [Errno 2] No such file or directory: 'hdfs'

## 8. Llamar al API de fraude con muestras del dataset

In [ ]:
import requests
import json

API_URL = "http://ai-service:8000/predict"  # nombre del servicio en shared-ml-network

# Seleccionar 5 muestras del dataset procesado
sample_rows = sdf_features.select(
    "amount", "old_balance_orig", "new_balance_orig",
    "old_balance_dest", "new_balance_dest",
    "orig_out_degree", "orig_pagerank", "orig_community",
    "dest_in_degree", "dest_pagerank", "dest_community",
    "type_CASH_IN", "type_CASH_OUT", "type_DEBIT",
    "type_PAYMENT", "type_TRANSFER"
).limit(5).toPandas()

results = []
for _, row in sample_rows.iterrows():
    payload = row.to_dict()
    try:
        resp = requests.post(API_URL, json=payload, timeout=5)
        resp.raise_for_status()
        results.append(resp.json())
    except Exception as e:
        results.append({"error": str(e)})

print("Resultados del API para 5 transacciones:")
for i, r in enumerate(results):
    print(f"  [{i+1}] {r}")

## 9. Guardar predicciones en HDFS

In [ ]:
import requests

# Broadcast de la URL para usar en UDF distribuida
from pyspark.sql.functions import udf
from pyspark.sql.types import DoubleType, StructType, StructField

API_URL_LOCAL = "http://ai-service:8000/predict"

def predict_fraud(amount, old_bal_orig, new_bal_orig, old_bal_dest, new_bal_dest,
                  out_deg, pr_orig, comm_orig, in_deg, pr_dest, comm_dest,
                  ci, co, deb, pay, tr):
    payload = {
        "amount": float(amount or 0),
        "old_balance_orig": float(old_bal_orig or 0),
        "new_balance_orig": float(new_bal_orig or 0),
        "old_balance_dest": float(old_bal_dest or 0),
        "new_balance_dest": float(new_bal_dest or 0),
        "orig_out_degree": float(out_deg or 0),
        "orig_pagerank": float(pr_orig or 0),
        "orig_community": float(comm_orig or 0),
        "dest_in_degree": float(in_deg or 0),
        "dest_pagerank": float(pr_dest or 0),
        "dest_community": float(comm_dest or 0),
        "type_CASH_IN": float(ci or 0),
        "type_CASH_OUT": float(co or 0),
        "type_DEBIT": float(deb or 0),
        "type_PAYMENT": float(pay or 0),
        "type_TRANSFER": float(tr or 0),
    }
    try:
        r = requests.post(API_URL_LOCAL, json=payload, timeout=3)
        return float(r.json().get("fraud_probability", -1))
    except:
        return -1.0

predict_udf = udf(predict_fraud, DoubleType())

# Aplicar al primer batch (1000 filas para no saturar)
sdf_batch = sdf_features.limit(1000)
sdf_with_pred = sdf_batch.withColumn(
    "fraud_probability",
    predict_udf(
        "amount", "old_balance_orig", "new_balance_orig",
        "old_balance_dest", "new_balance_dest",
        "orig_out_degree", "orig_pagerank", "orig_community",
        "dest_in_degree", "dest_pagerank", "dest_community",
        "type_CASH_IN", "type_CASH_OUT", "type_DEBIT",
        "type_PAYMENT", "type_TRANSFER"
    )
)

RESULTS_PATH = f"{DATA_ROOT}/data/fraud-results/batch_predictions.parquet"
if not HDFS_UP:
    os.makedirs(f"{LOCAL_DATA}/data/fraud-results", exist_ok=True)
sdf_with_pred.write.mode("overwrite").parquet(RESULTS_PATH)
print(f"[OK] Predicciones guardadas en {RESULTS_PATH}")

# Resumen de fraudes detectados
fraud_count = sdf_with_pred.filter(col("fraud_probability") >= 0.15).count()

total = sdf_with_pred.count()print(f"Fraudes detectados: {fraud_count}/{total} ({100*fraud_count/total:.1f}%)")

## 10. Verificación final del estado de HDFS

In [ ]:
import os

print(f"=== Estructura de datos ({DATA_ROOT}) ===")

if HDFS_UP:
    # Listar usando la API Java de HDFS
    fs = spark._jvm.org.apache.hadoop.fs.FileSystem.get(spark._jsc.hadoopConfiguration())
    for path in ["/data/raw", "/data/processed", "/data/fraud-results"]:
        try:
            files = fs.listStatus(spark._jvm.org.apache.hadoop.fs.Path(path))
            print(f"\n{path}/")
            for f in files:
                size_mb = f.getLen() / 1024 / 1024
                print(f"  {f.getPath().getName()}  ({size_mb:.2f} MB)")
        except Exception as e:
            print(f"\n{path}/ → no existe aún o error: {e}")
else:
    # Listar en sistema de archivos local
    for subdir in ["raw", "processed", "fraud-results"]:
        path = os.path.join(LOCAL_DATA, "data", subdir)
        if os.path.exists(path):
            print(f"\n{path}/")
            for f in os.listdir(path):
                size_mb = os.path.getsize(os.path.join(path, f)) / 1024 / 1024
                print(f"  {f}  ({size_mb:.2f} MB)")
        else:
            print(f"\n{path}/ → no existe aún")

spark.stop()
print("\nSparkSession cerrada. Ingesta completada.")
